## Set up and Imports

In [ ]:
#  Standard library 
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

#  Add src/ to Python path 
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

#  Data manipulation 
import pandas as pd
import numpy as np

#  Statistics 
# scipy.stats gives us the Kruskal-Wallis test — a non-parametric test
# that checks whether the spend distributions across channels are
# statistically different from each other, without assuming normality
from scipy import stats

#  Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

#  Shared utilities 
from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '04_geographic_clv_analysis')

print('All imports successful')
print(f'Project root: {project_root}')

## Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

# Load orders with segment labels
df_raw = pd.read_csv(processed_path / 'orders_with_segments.csv')
df_raw['PURCHASE_TS'] = pd.to_datetime(df_raw['PURCHASE_TS'], errors='coerce')
df = df_raw.copy()

# Load RFM table — one row per customer with segment labels
rfm_raw = pd.read_csv(processed_path / 'rfm_segments.csv')
rfm = rfm_raw.copy()

print(f'Orders loaded:    {len(df):,}')
print(f'Customers loaded: {len(rfm):,}')
print(f'\nRegions in dataset:')
print(df['REGION'].value_counts(dropna=False))
print(f'\nUnique countries: {df["COUNTRY_CODE"].nunique()}')

In [ ]:
log(f'Orders loaded:    {len(df):,}')
log(f'Customers loaded: {len(rfm):,}')
log(f'\nRegions in dataset:')
log(df['REGION'].value_counts(dropna=False))
log(f'\nUnique countries: {df["COUNTRY_CODE"].nunique()}')

In [ ]:
#  Filter to analysis-ready orders 
# Exclude $0 orders and records with no region assigned
df_analysis = df[
    (df['USD_PRICE'] > 0) &
    (df['REGION'].notna()) &
    (df['REGION'] != 'Other')
].copy()

print(f'Orders after filtering: {len(df_analysis):,}')
print(f'Regions remaining:      {df_analysis["REGION"].nunique()}')
print(f'\nOrders per region:')
print(df_analysis['REGION'].value_counts())

In [ ]:
log(f'Orders after filtering: {len(df_analysis):,}')
log(f'Regions remaining:      {df_analysis["REGION"].nunique()}')
log(f'\nOrders per region:')
log(df_analysis['REGION'].value_counts())

In [ ]:
# Build a customer-level geographic table
# We work at customer level to avoid double-counting repeat buyers
# A customer with 3 orders should count once with their total spend

# Safe mode function — returns NaN instead of crashing on empty groups
def most_frequent(x):
    counts = x.value_counts()
    return counts.index[0] if len(counts) > 0 else np.nan

customer_geo = df_analysis.groupby('USER_ID').agg(
    country_code = ('COUNTRY_CODE', most_frequent),
    region       = ('REGION',       most_frequent)
).reset_index()

# Drop customers where country or region could not be determined
before = len(customer_geo)
customer_geo = customer_geo.dropna(subset=['country_code', 'region'])
after  = len(customer_geo)
print(f'Dropped {before - after} customers with no country/region data')

# Merge with RFM data
customer_geo = customer_geo.merge(
    rfm[['USER_ID', 'monetary', 'frequency', 'recency', 'RFM_score', 'segment']],
    on='USER_ID', how='inner'
)

print(f'Customer-geography table: {len(customer_geo):,} customers')
print(f'Regions covered:          {customer_geo["region"].nunique()}')
print(f'Countries covered:        {customer_geo["country_code"].nunique()}')

In [ ]:
log(f'Dropped {before - after} customers with no country/region data')
log(f'Customer-geography table: {len(customer_geo):,} customers')
log(f'Regions covered:          {customer_geo["region"].nunique()}')
log(f'Countries covered:        {customer_geo["country_code"].nunique()}')

## Regional CLV analysis

In [ ]:
# Regional CLV metrics
region_clv = customer_geo.groupby('region').agg(
    customers          = ('USER_ID',    'count'),
    total_revenue      = ('monetary',   'sum'),
    avg_spend          = ('monetary',   'mean'),
    median_spend       = ('monetary',   'median'),
    avg_rfm_score      = ('RFM_score',  'mean'),
    avg_frequency      = ('frequency',  'mean'),
    avg_recency        = ('recency',    'mean')
).round(2)

region_clv['revenue_share_pct']    = (
    region_clv['total_revenue'] / region_clv['total_revenue'].sum() * 100
).round(1)
region_clv['customer_share_pct']   = (
    region_clv['customers'] / region_clv['customers'].sum() * 100
).round(1)
region_clv['revenue_per_customer'] = (
    region_clv['total_revenue'] / region_clv['customers']
).round(2)

region_clv = region_clv.sort_values('avg_spend', ascending=False)

print(' Regional CLV Summary ')
print(region_clv.to_string())

In [ ]:
log(' Regional CLV Summary ')
log(region_clv.to_string())

In [ ]:
# Segment mix by region
seg_by_region = customer_geo.groupby(
    ['region', 'segment']
).size().unstack(fill_value=0)

seg_by_region_pct = seg_by_region.div(seg_by_region.sum(axis=1), axis=0) * 100
seg_by_region_pct = seg_by_region_pct.round(1)

# High value rate per region
high_value_segs = ['Champions', 'Loyal Customers']
available_segs  = [s for s in high_value_segs if s in seg_by_region_pct.columns]
region_quality  = seg_by_region_pct[available_segs].sum(axis=1).sort_values(ascending=False)

print(' Segment mix by region (%) ')
print(seg_by_region_pct.to_string())
print()
print(' High-value customer rate by region ')
print(region_quality.to_string())

In [ ]:
log(' Segment mix by region (%) ')
log(seg_by_region_pct.to_string())
log()
log(' High-value customer rate by region ')
log(region_quality.to_string())

## Country Level Analysis

In [ ]:
# Get top 20 countries by customer count
top_countries = customer_geo['country_code'].value_counts().head(20).index.tolist()

country_clv = customer_geo[customer_geo['country_code'].isin(top_countries)]\
    .groupby('country_code').agg(
        customers     = ('USER_ID',   'count'),
        avg_spend     = ('monetary',  'mean'),
        median_spend  = ('monetary',  'median'),
        total_revenue = ('monetary',  'sum'),
        avg_rfm_score = ('RFM_score', 'mean')
    ).round(2)

country_clv['revenue_share_pct']  = (
    country_clv['total_revenue'] / customer_geo['monetary'].sum() * 100
).round(1)
country_clv['customer_share_pct'] = (
    country_clv['customers'] / len(customer_geo) * 100
).round(1)

# ROI ratio: revenue share / customer share
# > 1.0 means the country generates more revenue than its customer share warrants
# This identifies markets that punch above their weight
country_clv['roi_ratio'] = (
    country_clv['revenue_share_pct'] / country_clv['customer_share_pct']
).round(2)

country_clv = country_clv.sort_values('avg_spend', ascending=False)

print(' Top 20 Countries by CLV Metrics ')
print(country_clv[['customers', 'avg_spend', 'median_spend',
                   'revenue_share_pct', 'customer_share_pct', 'roi_ratio']].to_string())

In [ ]:
log(' Top 20 Countries by CLV Metrics ')
log(country_clv[['customers', 'avg_spend', 'median_spend',
                   'revenue_share_pct', 'customer_share_pct', 'roi_ratio']].to_string())

## Bootstrap Confidence Intervals

In [ ]:
def bootstrap_ci(data, n_bootstrap=1000, ci=95, random_state=42):
    """
    Calculate bootstrap confidence interval for the mean.

    Parameters
    ----------
    data         : array-like of values
    n_bootstrap  : number of resamples (1000 is standard)
    ci           : confidence level (95 = 95% CI)
    random_state : for reproducibility

    Returns
    -------
    (mean, lower_bound, upper_bound)
    """
    rng      = np.random.default_rng(random_state)
    data_arr = np.array(data)
    means    = [
        rng.choice(data_arr, size=len(data_arr), replace=True).mean()
        for _ in range(n_bootstrap)
    ]
    lower = np.percentile(means, (100 - ci) / 2)
    upper = np.percentile(means, 100 - (100 - ci) / 2)
    return np.mean(data_arr), lower, upper


# Calculate bootstrap CIs for each region
print(' Bootstrap 95% Confidence Intervals — Avg Spend by Region ')
print(f'{"Region":<12} {"N":>6} {"Mean":>10} {"Lower 95%":>12} {"Upper 95%":>12} {"CI Width":>10}')
print('-' * 64)

ci_results = {}
for region in sorted(customer_geo['region'].unique()):
    region_data = customer_geo[customer_geo['region'] == region]['monetary']
    if len(region_data) < 10:
        continue
    mean, lower, upper = bootstrap_ci(region_data)
    ci_results[region] = {'mean': mean, 'lower': lower, 'upper': upper,
                          'n': len(region_data), 'width': upper - lower}
    print(f'{region:<12} {len(region_data):>6,} {mean:>10.2f} {lower:>12.2f} {upper:>12.2f} {upper-lower:>10.2f}')

In [ ]:
# Kruskal-Wallis test across regions
region_groups = [
    group['monetary'].values
    for _, group in customer_geo.groupby('region')
    if len(group) >= 10
]

stat, p_value = stats.kruskal(*region_groups)
print(' Kruskal-Wallis Test: Customer Spend by Region ')
print(f'  H statistic: {stat:.4f}')
print(f'  p-value:     {p_value:.6f}')
print()
if p_value < 0.05:
    print('  RESULT: p < 0.05 → spend differences across regions are')
    print('  statistically significant — not due to random chance.')
else:
    print('  RESULT: p >= 0.05 → differences are not statistically significant.')
print()

# Pairwise tests between regions
regions      = sorted(customer_geo['region'].unique())
n_tests      = len(regions) * (len(regions) - 1) / 2
bonferroni_t = 0.05 / n_tests

print(f'Pairwise Mann-Whitney U tests ({int(n_tests)} pairs)')
print(f'Bonferroni threshold: {bonferroni_t:.4f}')
print()
print(f'{"Region A":<12} {"Region B":<12} {"Median A":>10} {"Median B":>10} {"p-value":>12} {"Significant?":>14}')
print('-' * 72)

for i, r_a in enumerate(regions):
    for r_b in regions[i+1:]:
        g_a = customer_geo[customer_geo['region'] == r_a]['monetary']
        g_b = customer_geo[customer_geo['region'] == r_b]['monetary']
        if len(g_a) < 5 or len(g_b) < 5:
            continue
        _, p = stats.mannwhitneyu(g_a, g_b, alternative='two-sided')
        sig  = 'YES' if p < bonferroni_t else 'No'
        print(f'{r_a:<12} {r_b:<12} {g_a.median():>10.2f} {g_b.median():>10.2f} {p:>12.6f} {sig:>14}')

## Visualisations

In [ ]:
#  Chart 1: Average spend by region with bootstrap CIs 
# Error bars show the 95% confidence interval from bootstrapping
# Overlapping error bars suggest the difference may not be meaningful

ci_df = pd.DataFrame(ci_results).T.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

yerr_lower = ci_df['mean'] - ci_df['lower']
yerr_upper = ci_df['upper'] - ci_df['mean']

bars = ax.bar(
    ci_df.index,
    ci_df['mean'],
    color='#2E75B6', alpha=0.85,
    yerr=[yerr_lower, yerr_upper],
    capsize=6,
    error_kw={'ecolor': '#1F4E79', 'linewidth': 1.5}
)

for bar, val in zip(bars, ci_df['mean']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + ci_df.loc[ci_df.index[list(ci_df['mean']).index(val)], 'upper'] - val + 5,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

ax.set_title('Average customer spend by region\n'
             'Error bars = 95% bootstrap confidence interval',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Average total customer spend (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

save_figure(fig, '02_04_avg_spend_by_region_ci.png')
plt.show()

In [ ]:
#  Chart 2: Revenue share vs customer share by region 
# Same approach as Notebook 3 but at regional level

fig, ax = plt.subplots(figsize=(10, 5))

regions_ordered = region_clv.sort_values('total_revenue', ascending=False).index.tolist()
x     = np.arange(len(regions_ordered))
width = 0.35

cust_shares = region_clv.loc[regions_ordered, 'customer_share_pct'].values
rev_shares  = region_clv.loc[regions_ordered, 'revenue_share_pct'].values

b1 = ax.bar(x - width/2, cust_shares, width, label='% of customers',
            color='#2E75B6', alpha=0.85)
b2 = ax.bar(x + width/2, rev_shares,  width, label='% of revenue',
            color='#70AD47', alpha=0.85)

for bar, val in zip(b1, cust_shares):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(b2, rev_shares):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(regions_ordered)
ax.set_title('Customer share vs revenue share by region',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Share (%)')
ax.legend()

save_figure(fig, '02_04_revenue_vs_customer_share_region.png')
plt.show()

In [ ]:
#  Chart 3: Top 20 countries — avg spend horizontal bar 
# Countries ranked by average spend per customer
# Coloured by whether they are above or below the overall average

fig, ax = plt.subplots(figsize=(12, 8))

overall_avg = customer_geo['monetary'].mean()
country_plot = country_clv.sort_values('avg_spend', ascending=True)

colors = ['#C00000' if v < overall_avg else '#2E75B6'
          for v in country_plot['avg_spend']]

bars = ax.barh(country_plot.index, country_plot['avg_spend'],
               color=colors, alpha=0.85)

ax.axvline(overall_avg, color='#404040', linestyle='--', linewidth=1.5,
           label=f'Overall avg: ${overall_avg:,.0f}')

for bar, val, n in zip(bars, country_plot['avg_spend'], country_plot['customers']):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}  (n={n:,})', va='center', fontsize=8)

ax.set_title('Average customer spend — top 20 countries\n'
             '(Blue = above average, Red = below average)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Average total customer spend (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

save_figure(fig, '02_04_avg_spend_by_country.png')
plt.show()

In [ ]:
#  Chart 4: ROI ratio by country 
# ROI ratio = revenue share / customer share
# > 1.0 = country generates more revenue than its customer share warrants
# < 1.0 = country underperforms relative to its customer count

fig, ax = plt.subplots(figsize=(12, 7))

roi_plot = country_clv.sort_values('roi_ratio', ascending=True)
colors   = ['#C00000' if v < 1.0 else '#2E75B6' for v in roi_plot['roi_ratio']]

bars = ax.barh(roi_plot.index, roi_plot['roi_ratio'], color=colors, alpha=0.85)

ax.axvline(1.0, color='#404040', linestyle='--', linewidth=1.5,
           label='Break-even (1.0)')

for bar, val in zip(bars, roi_plot['roi_ratio']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}x', va='center', fontsize=9)

ax.set_title('Revenue ROI ratio by country\n'
             '(>1.0 = punches above its weight | <1.0 = underperforms)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('ROI ratio (revenue share ÷ customer share)')
ax.legend()

save_figure(fig, '02_04_roi_ratio_by_country.png')
plt.show()

In [ ]:
#  Chart 5: Segment mix by region ─
seg_colors_map = {
    'Champions':      '#1F4E79',
    'Loyal Customers':'#2E75B6',
    'At Risk':        '#ED7D31',
    'Lapsed':         '#C00000'
}

seg_order  = ['Champions', 'Loyal Customers', 'At Risk', 'Lapsed']
available  = [s for s in seg_order if s in seg_by_region_pct.columns]

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(len(seg_by_region_pct))
for seg in available:
    vals = seg_by_region_pct[seg].values
    bars = ax.bar(seg_by_region_pct.index, vals, bottom=bottom,
                  label=seg, color=seg_colors_map.get(seg, 'grey'), alpha=0.85)
    for bar, val, bot in zip(bars, vals, bottom):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2, bot + val/2,
                    f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, color='white', fontweight='medium')
    bottom += vals

ax.set_title('Customer segment mix by region',
             fontsize=13, fontweight='medium')
ax.set_ylabel('% of region customers')
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1), fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

save_figure(fig, '02_04_segment_mix_by_region.png')
plt.show()

## Findings and Recommendations

In [ ]:
print('  NOTEBOOK 4 FINDINGS — GEOGRAPHIC CLV ANALYSIS')
print()
print('FINDING 1 — Regional differences are real but modest')
print('  APAC leads avg spend at $316, LATAM trails at $293.')
print('  The gap is only 7.8% — far smaller than the 68% gap')
print('  between best and worst channel found in Notebook 3.')
print('  Geography is a weaker CLV predictor than channel.')
print()
print('FINDING 2 — Bootstrap CIs reveal LATAM conclusions are uncertain')
print('  LATAM CI: $269-$319 (width: $49). The wide interval means')
print('  LATAM could plausibly match APAC or Americas in true avg')
print('  spend. Do not make investment decisions based on LATAM')
print("  alone — the sample (1,143 customers) isn't large enough")
print('  for confident conclusions.')
print()
print('FINDING 3 — Pairwise tests reveal the true structure')
print('  APAC vs LATAM: NOT significantly different (p=0.72)')
print('  EMEA vs LATAM: NOT significantly different (p=0.06)')
print('  The overall Kruskal-Wallis significance is mainly driven')
print('  by Americas vs other regions, not a clean hierarchy.')
print()
print('FINDING 4 — Japan is the standout high-value market')
print('  JP: $472 avg spend, 1.52x ROI ratio, 455 customers.')
print('  50% higher avg spend than the overall $310 average.')
print('  Japan generates 3.5% of revenue from 2.3% of customers.')
print('  Japan gaming culture (high console attach rates) explains')
print('  the premium spend behaviour — customers skew toward')
print('  high-ticket items like PS5 bundles and gaming laptops.')
print()
print('FINDING 5 — APAC top-heavy: JP, DK, KR drive the region average')
print('  Japan, Denmark and South Korea all have ROI ratios > 1.2x.')
print('  These three markets boost the APAC and EMEA averages.')
print('  Without them, the regional picture would be flatter.')
print()
print('FINDING 6 — Segment mix is uniform across all regions')
print('  Every region shows ~9% Champions, ~7% Loyal, ~43% At Risk,')
print('  ~40% Lapsed. Where a customer lives does not predict')
print('  their purchase behaviour segment. This rules out any')
print('  geography-specific segmentation strategy.')
print()
print('FINDING 7 — English-speaking markets underperform on CLV')
print('  AU (0.73x), CA (0.88x), IE (0.75x) all below break-even.')
print('  These are large, accessible markets but customers spend')
print('  less per head. Likely dominated by Nintendo Switch buyers')
print('  rather than high-ticket console or laptop purchasers.')
print()
print('RECOMMENDATIONS')
print()
print('  1. Invest in Japan retention above all other single markets')
print('     $472 avg spend, 1.52x ROI, statistically distinct from')
print('     all other markets. Japan customers are disproportionately')
print('     valuable — a targeted retention programme is justified.')
print()
print('  2. Target high-ticket products in JP, DK, KR specifically')
print('     These markets show a clear appetite for premium products.')
print('     PS5 bundles, gaming laptops, and high-end monitors should')
print('     be prioritised in marketing materials for these markets.')
print()
print('  3. Do not over-invest in AU, CA, IE on CLV basis')
print('     High English-speaking market volume is attractive but')
print('     ROI ratios below 0.9x mean these customers spend less')
print('     than their acquisition cost warrants. Review pricing')
print('     or product mix for these markets before scaling spend.')
print()
print('  4. Channel matters more than geography')
print('     The 68% spend gap between affiliate and email channels')
print('     dwarfs the 7.8% gap between best and worst region.')
print('     If forced to choose between optimising channel mix or')
print('     geographic targeting, fix channel allocation first.')
print()
print('Notebook 4 complete ✓')
print('Figures saved     → reports/figures/  (5 charts)')
print('Exports saved     → data/processed/   (3 files)')
print('Ready for         → Project 02 Excel Report')

## Export

In [ ]:
os.makedirs(processed_path, exist_ok=True)

# Export regional CLV summary
region_output = region_clv.copy()
region_output['high_value_customer_pct'] = region_quality
region_output.to_csv(processed_path / 'regional_clv_summary.csv')
print('Exported: regional_clv_summary.csv')

# Export country CLV summary
country_clv.to_csv(processed_path / 'country_clv_summary.csv')
print('Exported: country_clv_summary.csv')

# Export bootstrap CI results
ci_export = pd.DataFrame(ci_results).T
ci_export.to_csv(processed_path / 'regional_bootstrap_ci.csv')
print('Exported: regional_bootstrap_ci.csv')

print()
print('All geographic CLV exports complete ✓')